# How to scale cell value according to contents

Instead of using the single region that contributes the most to the cell, use an average of all regions in the cell weighted by the proportion of their area that they contribute to it.

## Code setup

In [1]:
# Setting up paths to files:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

# Add path to package:
import sys
sys.path.insert(0,'../')
import isopods.convert

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    dir_data: str = '../data'
    grid_filename: str = 'lsoa2011_devon_1k_grid_df.csv'

paths = Paths()

## Load in grid data

In [3]:
# Proportions dataframe:
p = os.path.join(paths.dir_data, paths.grid_filename)
# Use converters to turn proportion data from strings to lists.
converters = {
    'label_props': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
    'area_props': lambda x: [float(f) for f in x[1:-1].split()]
}
df_props = pd.read_csv(p, index_col=0, converters=converters)

# Confirm that contents are as expected:
for col in ['label_props', 'area_props']:
    print(col)
    print('Data type:', type(df_props.iloc[0][col]))
    print('List value type:', type(df_props.iloc[0][col][0]))

label_props
Data type: <class 'list'>
List value type: <class 'str'>
area_props
Data type: <class 'list'>
List value type: <class 'float'>


In [4]:
df_props.head(3)

,cell_row,cell_col,label_props,area_props,prop_cell_covered,label_majority,overlapping_polys
cell_label,,,,,,,
109,109,0,[E01020288],[1.0],0.000,E01020288,0
110,110,0,[E01020288],[1.0],0.124,E01020288,0
111,111,0,[E01020288],[1.0],0.089,E01020288,0


Invent some data for the regions. Assign each a random integer:

In [5]:
lsoa_here = np.sort(df_props['label_props'].explode('label_props').unique())
dict_region_value = dict(zip(lsoa_here,
                             np.random.randint(0, 11, len(lsoa_here))))

Pick out the labels of cells that contain many regions:

In [6]:
inds_many_labels = df_props[df_props['label_props'].str.len() > 3].index

## Scaled values

In [7]:
df_props['majority_area_value'] = (
    df_props['label_majority'].map(dict_region_value))

df_scaled_values = isopods.convert.calculate_scaled_cell_values(
    df_props, dict_region_value)
df_props = pd.concat((df_props, df_scaled_values), axis='columns')

Check the values for a cell that contains multiple regions:

In [8]:
cols_to_show = ['label_props', 'area_props', 'majority_area_value',
                'area_value_scaled']
display(df_props.loc[inds_many_labels[:5], cols_to_show])

,label_props,area_props,majority_area_value,area_value_scaled
cell_label,,,,
3682,"[E01015142, E01015083, E01015150, E01015146, E...","[0.779, 0.178, 0.042, 0.0, 0.0]",3,3.227
3751,"[E01020300, E01020295, E01020283, E01020281]","[0.52, 0.286, 0.14, 0.054]",6,5.092
3752,"[E01020310, E01020295, E01020298, E01020300, E...","[0.686, 0.1, 0.09, 0.062, 0.062]",5,4.784
3799,"[E01015145, E01015146, E01015039, E01015143, E...","[0.526, 0.276, 0.092, 0.056, 0.024, 0.017, 0.0...",8,7.223
3800,"[E01015150, E01015083, E01015144, E01015149, E...","[0.281, 0.225, 0.155, 0.112, 0.1, 0.095, 0.025...",0,4.055
